# MedCLIP-SAMv2 Text+Boxes — Augmented Dataset (Lambda)

Applies **MedCLIP-SAMv2** (BiomedCLIP saliency + MuscleMap WB bounding box → MedSAM)
to the 20 augmented NIfTI water volumes.

⚠️ **Run `lambda_musclemap_wb_augmented.ipynb` first** — this notebook uses the WB
segmentations as bounding-box prompts.

MuscleMap WB saves augmented outputs as `{stem}_augmented000_water_dseg.nii.gz`.
For each water file `{stem}_augmented000_water.nii.gz` the MM seg is looked up as
`{stem}_augmented000_water_dseg.nii.gz` in `~/musclemap_wb_augmented_segs/`.

Data: `~/our_augmented_dataset/{stem}_augmented000_water.nii.gz`
MM WB segs: `~/musclemap_wb_augmented_segs/{stem}_augmented000_water_dseg.nii.gz`
Output: `~/medclipsamv2_textboxes_augmented_segs/{stem}_mcsam2textboxes.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/our_augmented_dataset/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/our_augmented_dataset/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/musclemap_wb_augmented_segs/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/musclemap_wb_augmented_segs/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/path/to/medsam_vit_b.pth" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsam_vit_b.pth
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medclipsamv2_textboxes_augmented_segs/ \
  /path/to/local/medclipsamv2textboxes/augmented_segs/
```
**Terminate the instance when done.**

In [ ]:
# ── Install SAM in the main Jupyter kernel (used for MedSAM inference) ───────
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'SimpleITK', 'scikit-image', 'opencv-python-headless'])

import torch
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# ── Clone MedCLIP-SAMv2 repo and build a venv for BiomedCLIP saliency ────────
import os

REPO_DIR = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_DIR = os.path.expanduser('~/mcsam2_env')
VENV_PY  = os.path.join(VENV_DIR, 'bin', 'python')

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/HealthX-Lab/MedCLIP-SAMv2.git', REPO_DIR])
    print('Cloned MedCLIP-SAMv2')
else:
    print('Repo already present')

if not os.path.exists(VENV_PY):
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])
    print('Venv created')
else:
    print('Venv already exists')

def venv_pip(*args):
    subprocess.check_call([VENV_PY, '-m', 'pip'] + list(args))

venv_pip('install', '-q', '--upgrade', 'pip')
venv_pip('install', '-q', 'numpy', 'scikit-learn')
venv_pip('install', '-q', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124')
venv_pip('install', '-q', '-e', os.path.join(REPO_DIR, 'segment-anything'))
venv_pip('install', '-q',
    'git+https://github.com/lucasb-eyer/pydensecrf.git')
venv_pip('install', '-q',
    'open_clip_torch', 'opencv-python', 'SimpleITK', 'Pillow',
    'huggingface_hub', 'transformers<4.46',
    'matplotlib', 'grad-cam', 'pandas', 'tqdm', 'scipy')
print('Venv dependencies installed.')

In [ ]:
# ── Paths and configuration ───────────────────────────────────────────────────
import glob, shutil, tempfile
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
import cv2
from PIL import Image
from skimage import transform
from segment_anything import sam_model_registry

REPO_DIR    = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_PY     = os.path.expanduser('~/mcsam2_env/bin/python')
VENV_DIR    = os.path.expanduser('~/mcsam2_env')
MEDSAM_CKPT = os.path.expanduser('~/medsam_vit_b.pth')
DATA_DIR    = os.path.expanduser('~/our_augmented_dataset')
MM_DIR      = os.path.expanduser('~/musclemap_wb_augmented_segs')
OUTPUT_DIR  = os.path.expanduser('~/medclipsamv2_textboxes_augmented_segs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAL_SCRIPT = os.path.join(REPO_DIR, 'saliency_maps', 'generate_saliency_maps.py')

# MedSAM runs on CPU to avoid OOM when also holding saliency data
SAM_DEVICE = 'cpu'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

for label, path in [
    ('MedSAM ckpt',   MEDSAM_CKPT),
    ('MM WB segs',    MM_DIR),
    ('Saliency script', SAL_SCRIPT),
]:
    ok = os.path.exists(path)
    print(f'  {"OK" if ok else "MISSING"}: {label} ({path})')
    if not ok and label in ('MedSAM ckpt', 'MM WB segs'):
        raise FileNotFoundError(f'{label} not found — upload it first')

nii_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))
print(f'Found : {len(nii_files)} water NIfTI volumes')

In [ ]:
# ── Muscle definitions: output key, text prompt, MuscleMap WB label index ────
MUSCLES = [
    ('R_gracilis',
     'gracilis muscle right thigh Dixon MRI axial cross section',
     7152),
    ('L_gracilis',
     'gracilis muscle left thigh Dixon MRI axial cross section',
     7151),
    ('R_sartorius',
     'sartorius muscle right thigh Dixon MRI axial cross section',
     7142),
    ('L_sartorius',
     'sartorius muscle left thigh Dixon MRI axial cross section',
     7141),
]
print('Muscles:', [m[0] for m in MUSCLES])

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
import glob as _glob
_venv_site = _glob.glob(os.path.join(VENV_DIR, 'lib', 'python3.*', 'site-packages'))
VENV_SITE = _venv_site[0] if _venv_site else ''
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH']       = VENV_SITE + ':' + os.environ.get('PYTHONPATH', '')
SUBPROCESS_ENV['PYTHONNOUSERSITE'] = '1'
SUBPROCESS_ENV['MPLBACKEND']       = 'Agg'


# ── numpy <-> torch bridges that survive has_numpy=False ─────────────────────
# torch is imported (cell 2) before numpy is ever imported in this kernel, so
# PyTorch's has_numpy flag latches to False and never re-checks. Any later
# call to torch.from_numpy() / tensor.numpy() then raises "Numpy is not
# available", even though numpy itself works fine. frombuffer/untyped_storage
# go through the buffer protocol instead, which isn't gated by that check.

def _np_to_t(arr: np.ndarray) -> torch.Tensor:
    a = np.ascontiguousarray(arr, dtype=np.float32)
    return torch.frombuffer(a, dtype=torch.float32).clone().reshape(a.shape)


def _t_to_np(t: torch.Tensor) -> np.ndarray:
    t_con = t.detach().cpu().float().clone().contiguous()
    raw   = bytes(t_con.untyped_storage())
    return np.frombuffer(raw, dtype=np.float32).reshape(t_con.shape).copy()


def export_slices_as_png(img_array, out_dir):
    """Write (D, H, W) float volume as 0-indexed RGB PNGs for saliency script."""
    os.makedirs(out_dir, exist_ok=True)
    for i in range(img_array.shape[0]):
        sl = img_array[i]
        sl_norm  = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        sl_uint8 = (sl_norm * 255).astype(np.uint8)
        Image.fromarray(np.stack([sl_uint8] * 3, axis=-1)).save(
            os.path.join(out_dir, f'{i}.png')
        )


def run_saliency(png_dir, sal_dir, text_prompt):
    """Run BiomedCLIP GradCAM saliency generation via the venv subprocess."""
    os.makedirs(sal_dir, exist_ok=True)
    result = subprocess.run(
        [VENV_PY, SAL_SCRIPT,
         '--input-path',  png_dir,
         '--output-path', sal_dir,
         '--val-path',    png_dir,
         '--model-name',  'BiomedCLIP',
         '--device',      DEVICE],
        input=text_prompt + '\n',
        text=True, capture_output=True,
        cwd=REPO_DIR, env=SUBPROCESS_ENV,
    )
    if result.returncode != 0:
        print('SALIENCY STDERR:', result.stderr[-1000:])
        raise RuntimeError(f'Saliency failed for prompt: {text_prompt}')


def load_saliency_volume(sal_dir, num_slices, target_size=256):
    """
    Load per-slice saliency PNGs into a (D, 1, target_size, target_size) float32
    tensor scaled to SAM logit range [-6, +6].
    """
    vol = np.zeros((num_slices, target_size, target_size), dtype=np.float32)
    for i in range(num_slices):
        png_path = os.path.join(sal_dir, f'{i}.png')
        if os.path.exists(png_path):
            s = cv2.imread(png_path, cv2.IMREAD_GRAYSCALE)
            if s is not None:
                s = cv2.resize(s, (target_size, target_size),
                               interpolation=cv2.INTER_LINEAR).astype(np.float32)
                vol[i] = (s / 255.0) * 12.0 - 6.0
    return _np_to_t(vol[:, None, :, :])  # (D, 1, 256, 256)


def get_mm_boxes(seg_array, mm_label, img_H, img_W, margin=5):
    """
    Extract per-slice bounding boxes for one MuscleMap label.
    Returns a list of length D; each entry is either a (4,) float array
    [x1, y1, x2, y2] in 1024-space, or None if the muscle is absent.
    """
    D = seg_array.shape[0]
    boxes = []
    for sl in range(D):
        mask = (seg_array[sl] == mm_label).astype(np.uint8)
        if not mask.any():
            boxes.append(None)
            continue
        rows = np.where(np.any(mask, axis=1))[0]
        cols = np.where(np.any(mask, axis=0))[0]
        r0, r1 = rows[[0, -1]]
        c0, c1 = cols[[0, -1]]
        H, W = mask.shape
        box_img = np.array([
            max(0, c0 - margin), max(0, r0 - margin),
            min(W - 1, c1 + margin), min(H - 1, r1 + margin),
        ], dtype=float)
        box_1024 = box_img / np.array([img_W, img_H, img_W, img_H]) * 1024
        boxes.append(box_1024)
    return boxes


def medsam_infer_with_saliency(sam_model, img_embed, box_1024, saliency_256,
                                H, W, device):
    """
    Run MedSAM with a bounding box prompt AND a dense saliency mask prompt.
    box_1024    : (4,) array [x1, y1, x2, y2] in 1024-space
    saliency_256: (1, 1, 256, 256) float tensor, logit-scaled from BiomedCLIP
    Returns     : (H, W) uint8 binary mask
    """
    box_t = torch.tensor(box_1024, dtype=torch.float, device=device)
    box_t = box_t[None, None, :]          # (1, 1, 4)
    sal_t = saliency_256.to(device)       # (1, 1, 256, 256)

    with torch.no_grad():
        sparse_emb, dense_emb = sam_model.prompt_encoder(
            points=None,
            boxes=box_t,
            masks=sal_t,
        )
        low_res_logits, _ = sam_model.mask_decoder(
            image_embeddings=img_embed,
            image_pe=sam_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False,
        )

    pred = F.interpolate(
        torch.sigmoid(low_res_logits), size=(H, W),
        mode='bilinear', align_corners=False,
    )
    return (_t_to_np(pred.squeeze()) > 0.5).astype(np.uint8)


print('Helpers defined.')

In [ ]:
# ── Load MedSAM once ──────────────────────────────────────────────────────────
sam_model = sam_model_registry['vit_b'](checkpoint=MEDSAM_CKPT)
sam_model.to(device=SAM_DEVICE)
sam_model.eval()
print('MedSAM loaded on', SAM_DEVICE)

In [ ]:
# ── Main processing loop ──────────────────────────────────────────────────────
for nii_path in nii_files:
    basename = os.path.basename(nii_path)
    stem     = basename.replace('_water.nii.gz', '')
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_mcsam2textboxes.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    # MuscleMap WB augmented saves output as {input_stem}_dseg.nii.gz
    # Input to MM was HV001_1_stack1_augmented000_water.nii.gz
    # → MM output is HV001_1_stack1_augmented000_water_dseg.nii.gz
    # Older runs of lambda_musclemap_wb_augmented.ipynb stripped "_water"
    # first, so fall back to that naming if the current one isn't found.
    seg_path = os.path.join(MM_DIR,
        os.path.basename(nii_path).replace('_water.nii.gz', '_water_dseg.nii.gz'))
    if not os.path.exists(seg_path):
        seg_path = os.path.join(MM_DIR, f'{stem}_dseg.nii.gz')
    if not os.path.exists(seg_path):
        print(f'  [skip] no MuscleMap WB seg for {stem} — run lambda_musclemap_wb_augmented first')
        continue

    print(f'\nProcessing: {stem}')
    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)  # (D, H, W)
    seg_array = sitk.GetArrayFromImage(sitk.ReadImage(seg_path)).astype(np.int32)
    D, H, W   = img_array.shape
    print(f'  Shape: {img_array.shape}  MM seg: {seg_array.shape}')

    tmp_root  = tempfile.mkdtemp(prefix='mctba_')
    all_masks = {}

    try:
        # ── Export slices once (reused for all muscles) ───────────────
        png_dir = os.path.join(tmp_root, 'slices')
        export_slices_as_png(img_array, png_dir)

        # ── Precompute SAM image embeddings once per slice ────────────
        print('  Computing SAM embeddings...')
        embeddings = []
        for sl_idx in range(D):
            sl        = img_array[sl_idx]
            img_norm  = sl * 255.0 / (sl.max() + 1e-8)
            img_3c    = np.repeat(img_norm[:, :, None], 3, axis=-1)
            img_1024  = transform.resize(
                img_3c, (1024, 1024), order=3,
                preserve_range=True, anti_aliasing=True,
            ).astype(np.uint8)
            img_1024  = (img_1024 - img_1024.min()) / np.clip(
                img_1024.max() - img_1024.min(), 1e-8, None
            )
            img_t = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(SAM_DEVICE)
            with torch.no_grad():
                emb = sam_model.image_encoder(img_t)
            embeddings.append(emb)
            del img_t
        print(f'  {D} embeddings ready')

        # ── Per-muscle: saliency + box → SAM ─────────────────────────
        for muscle_name, text_prompt, mm_label in MUSCLES:
            print(f'  [{muscle_name}] "{text_prompt}"')

            sal_dir = os.path.join(tmp_root, f'sal_{muscle_name}')

            # Stage 1: BiomedCLIP → per-slice saliency heatmaps
            run_saliency(png_dir, sal_dir, text_prompt)

            # Load all saliency PNGs as (D, 1, 256, 256) logit tensor
            sal_vol = load_saliency_volume(sal_dir, D)  # (D, 1, 256, 256)

            # MuscleMap bounding boxes per slice
            boxes = get_mm_boxes(seg_array, mm_label, H, W)

            vol_mask = np.zeros((D, H, W), dtype=np.uint8)

            for sl_idx in range(D):
                box = boxes[sl_idx]
                if box is None:
                    # No MuscleMap box on this slice — fall back to
                    # saliency-only: threshold at 0 (positive logit region)
                    sal_np = _t_to_np(sal_vol[sl_idx, 0])
                    sal_up = cv2.resize(
                        (sal_np > 0).astype(np.uint8), (W, H),
                        interpolation=cv2.INTER_NEAREST,
                    )
                    vol_mask[sl_idx] = sal_up
                    continue

                vol_mask[sl_idx] = medsam_infer_with_saliency(
                    sam_model,
                    embeddings[sl_idx],
                    box,
                    sal_vol[sl_idx:sl_idx+1],   # (1, 1, 256, 256)
                    H, W,
                    device=SAM_DEVICE,
                )

            all_masks[muscle_name] = vol_mask
            print(f'    {int(vol_mask.sum()):,} positive voxels')

        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved -> {out_path}')

    except Exception as exc:
        print(f'  ERROR on {stem}: {exc}')

    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)

print('\nAll done.')

In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────────
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Output files: {len(results)} / {len(nii_files)}')
if results:
    sample = np.load(results[0])
    print(f'  Sample: {os.path.basename(results[0])}')
    for k in sorted(sample.files):
        arr = sample[k]
        print(f'    {k}: shape={arr.shape}  voxels={int(arr.sum()):,}')